In [ ]:
# 12 — Silver: typed, validated, deduplicated incidents
# Reads new bronze rows incrementally (Delta stream, availableNow). Each micro-batch is cleaned with the
# shared rules in pipeline_lib.clean_incidents (the same rules as the Lakebase CHECK constraints):
# fixable issues are repaired, violations go to quarantine_incidents with reason codes, and valid rows are
# MERGEd into silver_incidents (latest version per id wins, so duplicates collapse and re-runs are safe).
# Live app changes reach silver through the real-time stream (20_live_stream) from Lakebase CDF.

In [ ]:
import os, sys, time
sys.path.insert(0, os.getcwd())          # pipeline_lib.py lives next to this notebook
import pipeline_lib as pl
from pyspark.sql import functions as F

dbutils.widgets.text("run_id", "manual")
RUN_ID = dbutils.widgets.get("run_id")
pl.ensure_tables(spark)

import cloudpickle
cloudpickle.register_pickle_by_value(pl)   # ship pipeline_lib with the foreachBatch function

def process(batch, batch_id):
    # Record the micro-batch size here: streaming progress is not always reported on serverless
    batch.sparkSession.createDataFrame([(batch_id, batch.count())], "batch_id BIGINT, n_rows BIGINT") \
        .withColumn("processed_at", F.current_timestamp()) \
        .write.mode("append").saveAsTable(pl.fq("_silver_batches"))
    valid, rejected = pl.clean_incidents(batch, "batch_export")
    rejected.write.mode("append").saveAsTable(pl.T_QUARANTINE)
    pl.merge_into_silver(batch.sparkSession, valid)

In [ ]:
started = time.time()
rejected_before = spark.table(pl.T_QUARANTINE).where("_source = 'batch_export'").count()

query = (spark.readStream.table(pl.T_BRONZE)
         .writeStream.foreachBatch(process)
         .option("checkpointLocation", f"{pl.CHECKPOINT_DIR}/silver")
         .trigger(availableNow=True)
         .start())
query.awaitTermination()

rows_in  = (spark.table(pl.fq("_silver_batches"))
            .where(F.col("processed_at") >= F.lit(started).cast("timestamp"))
            .agg(F.coalesce(F.sum("n_rows"), F.lit(0))).first()[0])
rejected = spark.table(pl.T_QUARANTINE).where("_source = 'batch_export'").count() - rejected_before
silver   = spark.table(pl.T_SILVER)
n_silver, n_ids = silver.count(), silver.select("id").distinct().count()
display(spark.table(pl.T_QUARANTINE).select(F.explode("reasons").alias("reason"))
        .groupBy("reason").count().orderBy(F.desc("count")))

pl.log_step(spark, RUN_ID, "12_silver", started, rows_in=rows_in, rows_out=n_silver, rows_rejected=rejected,
            checks={"one_row_per_incident": n_silver == n_ids,
                    "silver_has_most_rows": n_silver >= 980_000,
                    "defects_caught": rows_in == 0 or rejected > 0})